# 18: Measurement Simulation (Coleta com Erro)

## Objetivo
Simular uma coleta experimental de visibilidade interferométrica com erro de medição, avaliar a recuperabilidade dos parâmetros e verificar a distinguibilidade entre os regimes linear ($N$) e quadrático ($N^2$).

**Tag Epistemológica: PREVISÃO**

> [!IMPORTANT]
> Este notebook não usa dados reais de bancada. Ele gera dados sintéticos auditáveis para validar o desenho de coleta antes da fase experimental.

In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

# Parâmetros de referência (trilha de previsão existente)
tau = 1.0
a_ref = 0.014
b_ref = 0.00014

# Configuração de coleta (protocolo mínimo)
N_points = np.array([30, 40, 50, 60, 70])
runs_per_point = 10
sigma_measurement = 0.03  # erro absoluto ~3% em visibilidade (meta <5%)

print(f"seed={SEED} | tau={tau} | sigma={sigma_measurement} | runs={runs_per_point}")

seed=42 | tau=1.0 | sigma=0.03 | runs=10


In [2]:
def model_linear(N, a, tau):
    return np.exp(-a * N * tau / 2)


def model_quadratic(N, b, tau):
    return np.exp(-b * (N ** 2) * tau / 2)


def generate_measurements(N_values, runs, sigma, tau, b_true, rng):
    rows = []
    for N in N_values:
        v_true = model_quadratic(N, b_true, tau)
        for run in range(1, runs + 1):
            noise = rng.normal(0.0, sigma)
            v_obs = float(np.clip(v_true + noise, 1e-6, 1.0))
            rows.append({
                "N": int(N),
                "run": int(run),
                "V_true": float(v_true),
                "V_obs": v_obs,
                "noise": float(noise),
            })
    return pd.DataFrame(rows)


def fit_params_from_visibility(df, tau):
    grouped = df.groupby("N", as_index=False).agg(V_mean=("V_obs", "mean"), V_std=("V_obs", "std"))
    y = -2.0 * np.log(np.clip(grouped["V_mean"].to_numpy(), 1e-9, 1.0)) / tau
    N = grouped["N"].to_numpy(dtype=float)

    # Ajustes sem intercepto para manter consistência operacional com o protocolo
    a_hat = float(np.dot(N, y) / np.dot(N, N))
    N2 = N ** 2
    b_hat = float(np.dot(N2, y) / np.dot(N2, N2))

    grouped["V_pred_linear"] = model_linear(N, a_hat, tau)
    grouped["V_pred_quadratic"] = model_quadratic(N, b_hat, tau)

    grouped["err_linear"] = grouped["V_mean"] - grouped["V_pred_linear"]
    grouped["err_quadratic"] = grouped["V_mean"] - grouped["V_pred_quadratic"]

    rmse_linear = float(np.sqrt(np.mean(grouped["err_linear"] ** 2)))
    rmse_quadratic = float(np.sqrt(np.mean(grouped["err_quadratic"] ** 2)))

    return grouped, a_hat, b_hat, rmse_linear, rmse_quadratic


def classify_winner(rmse_linear, rmse_quadratic, margin=0.05):
    # margem relativa mínima para declarar distinção robusta
    rel_gain = (rmse_linear - rmse_quadratic) / max(rmse_linear, 1e-12)
    if rel_gain > margin:
        return "quadratic", rel_gain
    if -rel_gain > margin:
        return "linear", -rel_gain
    return "inconclusive", abs(rel_gain)

In [3]:
df = generate_measurements(
    N_values=N_points,
    runs=runs_per_point,
    sigma=sigma_measurement,
    tau=tau,
    b_true=b_ref,
    rng=rng,
)

grouped, a_hat, b_hat, rmse_lin, rmse_quad = fit_params_from_visibility(df, tau=tau)
winner, gain = classify_winner(rmse_lin, rmse_quad, margin=0.05)

print("RESUMO DA SIMULAÇÃO")
print("===================")
print(f"Coletas totais: {len(df)}")
print(f"Parâmetro estimado a (linear):     {a_hat:.6f}  (ref={a_ref:.6f})")
print(f"Parâmetro estimado b (quadrático): {b_hat:.6f}  (ref={b_ref:.6f})")
print(f"RMSE linear:     {rmse_lin:.6f}")
print(f"RMSE quadrático: {rmse_quad:.6f}")
print(f"Modelo vencedor: {winner} (ganho relativo={gain:.2%})")

grouped

RESUMO DA SIMULAÇÃO
Coletas totais: 50
Parâmetro estimado a (linear):     0.007834  (ref=0.014000)
Parâmetro estimado b (quadrático): 0.000136  (ref=0.000140)
RMSE linear:     0.034814
RMSE quadrático: 0.005945
Modelo vencedor: quadratic (ganho relativo=82.92%)


,N,V_mean,V_std,V_pred_linear,V_pred_quadratic,err_linear,err_quadratic
0,30,0.928876,0.028005,0.889136,0.940608,0.039741,-0.011732
1,40,0.902135,0.021708,0.854983,0.896864,0.047153,0.005272
2,50,0.842946,0.017209,0.822142,0.843597,0.020805,-0.000651
3,60,0.780323,0.030078,0.790562,0.782770,-0.010239,-0.002447
4,70,0.718728,0.013276,0.760196,0.716514,-0.041467,0.002214


In [4]:
# Auditoria de erro por ponto N
summary = df.groupby("N", as_index=False).agg(
    V_true=("V_true", "mean"),
    V_obs_mean=("V_obs", "mean"),
    V_obs_std=("V_obs", "std"),
)
summary["relative_error_pct"] = 100.0 * np.abs(summary["V_obs_mean"] - summary["V_true"]) / np.clip(summary["V_true"], 1e-9, None)
summary

,N,V_true,V_obs_mean,V_obs_std,relative_error_pct
0,30,0.938943,0.928876,0.028005,1.072177
1,40,0.894044,0.902135,0.021708,0.905013
2,50,0.839457,0.842946,0.017209,0.415649
3,60,0.777245,0.780323,0.030078,0.396082
4,70,0.709638,0.718728,0.013276,1.280914


## Conclusão Operacional

- A simulação gera uma coleta sintética reproduzível com erro controlado por semente fixa.
- O critério de seleção por RMSE permite avaliar se há distinguibilidade prática entre os modelos dentro da janela $N \in [30,70]$.
- Este artefato prepara a etapa de *fitting pipeline* com entradas mais realistas (dados com ruído), sem promover qualquer validação experimental.

## Próximo passo sugerido
Encadear este notebook ao pipeline de ajuste (MPV-55), incluindo:
1. bootstrap de incerteza dos parâmetros;
2. curva ROC para taxa de falso positivo na classificação linear vs quadrático;
3. sensibilidade a diferentes níveis de erro instrumental ($\sigma$).